# Regenerate ESM2 Embeddings Only
### Fast recovery notebook -- skips geNomad/CheckV since those results are already saved

Your Colab runtime was wiped (free-tier sessions don't persist once disconnected -- this is the exact tradeoff from dropping Drive). The good news: **geNomad and CheckV results are already safely saved** (`MASTER_prophage_validation_table.csv`, the virus/plasmid summary TSVs). We only lost `esm2_embeddings.npz`, the raw per-protein vectors -- everything else downstream was already summarized into CSVs that survived.

This notebook regenerates **only** what's needed to rebuild that file: extract prophage regions → extract cargo proteins → embed with ESM2 → **download the embeddings immediately** so this doesn't happen again. Should take well under an hour on a free T4, versus 1.5-3h for the full original pipeline.

**Upload when prompted:** `Final_Genomes.zip`, `PhiSpy_Results.zip` (same files as before).


## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())


## 2. Upload and unzip genome/PhiSpy inputs

In [ ]:
from google.colab import files
import os, zipfile

UPLOAD_DIR = "/content/uploaded_zips"
os.makedirs(UPLOAD_DIR, exist_ok=True)
print("Select Final_Genomes.zip and PhiSpy_Results.zip:")
uploaded = files.upload()
for fname in uploaded:
    dest = f"{UPLOAD_DIR}/{fname}"
    if os.path.exists(fname):
        os.rename(fname, dest)
    print(f"Saved: {dest}")


In [ ]:
WORK = "/content/work"
OUT  = "/content/outputs"
UNZIP_DIR = "/content/inputs"
for d in [WORK, OUT, UNZIP_DIR]:
    os.makedirs(d, exist_ok=True)

def find_or_unzip(zip_name, extract_subdir):
    target = f"{UNZIP_DIR}/{extract_subdir}"
    if not os.path.isdir(target) or not os.listdir(target):
        zpath = f"{UPLOAD_DIR}/{zip_name}"
        assert os.path.exists(zpath), f"Missing {zpath} -- re-run Cell 2 and upload {zip_name}"
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(target)
    cur = target
    while True:
        entries = [e for e in os.listdir(cur) if not e.startswith("__MACOSX")]
        subdirs = [e for e in entries if os.path.isdir(os.path.join(cur, e))]
        if len(subdirs) == 1 and not any(e.startswith(("GCA_", "GCF_")) for e in entries):
            cur = os.path.join(cur, subdirs[0])
        else:
            break
    return cur

GENOMES_DIR = find_or_unzip("Final_Genomes.zip", "Final_Genomes")
PHISPY_DIR  = find_or_unzip("PhiSpy_Results.zip", "PhiSpy_Results")

accessions = sorted(d for d in os.listdir(GENOMES_DIR)
                     if os.path.isdir(os.path.join(GENOMES_DIR, d)) and d.startswith(("GCA_", "GCF_"))
                     and os.path.exists(os.path.join(GENOMES_DIR, d, "genomic.fna")))
print(f"Found {len(accessions)} usable genome accessions")


## 3. Extract prophage nucleotide regions (needed to define region boundaries, matches earlier run)

In [ ]:
from Bio import SeqIO
import pandas as pd

COORD_COLS = ["prophage_id", "contig", "start", "stop", "attL_start", "attL_stop",
              "attR_start", "attR_stop", "attL_seq", "attR_seq", "note"]

def resolve_coord_file(acc):
    for c in [f"{PHISPY_DIR}/{acc}/{acc}_prophage_coordinates.tsv", f"{PHISPY_DIR}/{acc}/prophage_coordinates.tsv"]:
        if os.path.exists(c):
            return c
    return None

prophage_records = []
combined_fasta_path = f"{WORK}/prophage_regions.fna"
with open(combined_fasta_path, "w") as out_fh:
    for acc in accessions:
        coord_file = resolve_coord_file(acc)
        fna_file = f"{GENOMES_DIR}/{acc}/genomic.fna"
        if coord_file is None or not os.path.exists(fna_file):
            continue
        df = pd.read_csv(coord_file, sep="\t", header=None, names=COORD_COLS)
        if df.empty:
            continue
        genome_records = {r.id: r for r in SeqIO.parse(fna_file, "fasta")}
        for i, row in df.iterrows():
            s, e = int(row["start"]), int(row["stop"])
            if s > e: s, e = e, s
            contig_id = row["contig"]
            if contig_id not in genome_records:
                matches = [k for k in genome_records if str(contig_id) in k or k in str(contig_id)]
                contig_id = matches[0] if matches else None
            if contig_id is None:
                continue
            seq = genome_records[contig_id].seq[s-1:e]
            header = f"{acc}__pp{i+1}__{s}-{e}"
            out_fh.write(f">{header}\n{str(seq)}\n")
            prophage_records.append({"accession": acc, "prophage_id": f"pp{i+1}", "contig": contig_id,
                                      "start": s, "end": e, "length": e-s+1, "header": header})

meta_df = pd.DataFrame(prophage_records)
print(f"Extracted {len(meta_df)} prophage regions ({meta_df['accession'].nunique()} genomes) -- expect 65/20")


## 4. Extract prophage-encoded proteins from the NCBI annotation (same logic as before)

In [ ]:
from Bio import SeqIO as SeqIO2

OVERLAP_FRACTION = 0.5
protein_records = []
protein_fasta_path = f"{WORK}/prophage_proteins.faa"

with open(protein_fasta_path, "w") as out_fh:
    for acc, sub in meta_df.groupby("accession"):
        gbff_file = f"{GENOMES_DIR}/{acc}/genomic.gbff"
        if not os.path.exists(gbff_file):
            continue
        gb_records = {r.id: r for r in SeqIO2.parse(gbff_file, "genbank")}
        for _, region in sub.iterrows():
            contig, rs, re_ = region["contig"], region["start"], region["end"]
            if contig not in gb_records:
                continue
            gb_rec = gb_records[contig]
            for feat in gb_rec.features:
                if feat.type != "CDS":
                    continue
                fs, fe = int(feat.location.start) + 1, int(feat.location.end)
                overlap = max(0, min(fe, re_) - max(fs, rs))
                if overlap / (fe - fs + 1) < OVERLAP_FRACTION:
                    continue
                prot_seq = feat.qualifiers.get("translation", [None])[0]
                if not prot_seq:
                    continue
                locus_tag = feat.qualifiers.get("locus_tag", ["NA"])[0]
                product = feat.qualifiers.get("product", ["NA"])[0]
                pid = f"{region['header']}__{locus_tag}"
                out_fh.write(f">{pid}\n{prot_seq}\n")
                protein_records.append({"protein_id": pid, "accession": acc, "prophage_id": region["prophage_id"],
                                         "locus_tag": locus_tag, "product": product, "length_aa": len(prot_seq)})

prot_df = pd.DataFrame(protein_records)
print(f"Extracted {len(prot_df)} candidate prophage-encoded proteins -- expect ~2591")


## 5. ESM2 embeddings (GPU) -- the step that was actually lost

In [ ]:
!pip install -q fair-esm
import esm, torch, numpy as np

MODEL_NAME = "esm2_t12_35M_UR50D"
model, alphabet = esm.pretrained.load_model_and_alphabet(MODEL_NAME)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
batch_converter = alphabet.get_batch_converter()
NUM_LAYERS = model.num_layers

records = list(SeqIO2.parse(protein_fasta_path, "fasta"))
records.sort(key=lambda r: len(r.seq))

BATCH_SIZE = 16
CHECKPOINT_EVERY = 20
embeddings = {}
ckpt_path = f"{OUT}/esm2_embeddings.npz"

for b_start in range(0, len(records), BATCH_SIZE):
    batch = records[b_start:b_start + BATCH_SIZE]
    data = [(r.id, str(r.seq)[:1022]) for r in batch]
    labels, strs, tokens = batch_converter(data)
    tokens = tokens.to(device)
    with torch.no_grad():
        out = model(tokens, repr_layers=[NUM_LAYERS], return_contacts=False)
    reps = out["representations"][NUM_LAYERS]
    for i, (pid, seq) in enumerate(data):
        embeddings[pid] = reps[i, 1:len(seq)+1].mean(0).cpu().numpy()
    if (b_start // BATCH_SIZE) % CHECKPOINT_EVERY == 0:
        np.savez(ckpt_path, **embeddings)
        print(f"  checkpoint: {len(embeddings)} / {len(records)} embedded")

np.savez(ckpt_path, **embeddings)
print(f"Done: {len(embeddings)} embeddings saved to {ckpt_path}")


## 6. Download NOW -- don't close this tab until this finishes

This is the step that got skipped last time. Run this cell immediately and let the download complete before doing anything else.


In [ ]:
from google.colab import files as colab_files
colab_files.download(ckpt_path)
print("If the browser download didn't start automatically, check your browser's download-blocked notification.")
